# Action-focused SFT: NVIDIA trajectories, pilot và full success pool

Phần đầu kiểm tra **bộ pilot 1.000 trajectory đã serialize**. Đây chưa phải toàn bộ
success data dùng để chọn train. Phần **thống kê trajectories/task** bên dưới thống kê full success pool
hiện tại, giữ **50 task validation** và dùng **500 task SWE-bench Verified làm test cuối**.


In [2]:
import os
import json

In [3]:
from pathlib import Path

base_dir = str(next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "configs/datasets/nvidia_mini_swe_agent_v1_2_action_focused_sft.json").is_file()
))
base_dir


'/home/hqcfmlno1/Desktop/Workflow-Aware-Agent-Distillation'

In [6]:
processed_teacher_train_output = os.path.join(base_dir, 'data', 'processed', 'mini_swe_agent_v1_2_phase1', 'train.jsonl')

In [7]:
with open (processed_teacher_train_output, 'r', encoding = 'utf-8') as f:
    data_train = [json.loads(item) for item in f]
len(data_train)

1000

In [11]:
data_train[0].keys()

dict_keys(['repo', 'instance_id', 'trajectory_id', 'language', 'license', 'resolved', 'hf_dataset_name', 'category', 'teacher_model', 'shard', 'row_group', 'row_offset', 'task_key', 'messages', 'tools', 'checks', 'text', 'token_count', 'action_candidates', 'source_messages_sha256', 'command_availability'])

In [14]:
data_train[0]['text']

'<|im_start|>system\n# Tools\n\nYou have access to the following functions:\n\n<tools>\n{"type": "function", "function": {"name": "bash", "description": "Execute a bash command", "parameters": {"type": "object", "properties": {"command": {"type": "string", "description": "The bash command to execute"}}, "required": ["command"]}}}\n</tools>\n\nIf you choose to call a function ONLY reply in the following format with NO suffix:\n\n<tool_call>\n<function=example_function_name>\n<parameter=example_parameter_1>\nvalue_1\n</parameter>\n<parameter=example_parameter_2>\nThis is the value for the second parameter\nthat can span\nmultiple lines\n</parameter>\n</function>\n</tool_call>\n\n<IMPORTANT>\nReminder:\n- Function calls MUST follow the specified format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags\n- Required parameters MUST be specified\n- You may provide optional reasoning for your function call in natural language BEFORE the function c

In [21]:
len([i for i in data_train if i['token_count']<262144])

998

## Phân bố số success trajectories trên mỗi task

Đọc metadata từ `data/interim/nvidia_mini_swe_agent_v1_2_success/part-*.parquet`.
Bảng bên dưới chỉ thống kê **train_pool**, nhóm theo `task_key` (repository viết thường
+ số PR). Validation và các task trùng OmniCode không nằm trong bảng.

- `trajectories_per_task`: số successful trajectories của một task.
- `tasks`: số task có đúng số trajectory đó.
- `trajectories`: tổng trajectory của nhóm, bằng hai cột trên nhân nhau.

Giữ 50 task validation; test cuối là 500 task SWE-bench Verified.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Chạy được khi notebook mở từ repo root hoặc thư mục notebooks/.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "configs/datasets/nvidia_mini_swe_agent_v1_2_action_focused_sft.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Không tìm thấy repo root từ working directory hiện tại")

success_dir = project_root / "data/interim/nvidia_mini_swe_agent_v1_2_success"
parquet_files = sorted(success_dir.glob("part-*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Không có success Parquet tại {success_dir}")

metadata_columns = ["trajectory_id", "task_key", "partition", "repo", "resolved"]
metadata_parts = []
for parquet_file in parquet_files:
    part = pd.read_parquet(parquet_file, columns=metadata_columns)
    part["parquet_file"] = str(parquet_file.relative_to(project_root))
    # Vị trí trong file export; khác row_offset của shard NVIDIA nguồn.
    part["parquet_row"] = range(len(part))
    metadata_parts.append(part)
success_metadata = pd.concat(metadata_parts, ignore_index=True)
del metadata_parts, part

assert success_metadata["trajectory_id"].is_unique, "Trajectory ID bị trùng"
assert success_metadata["resolved"].eq(1).all(), "Có trajectory không pass"
assert success_metadata["task_key"].notna().all()
assert success_metadata.groupby("task_key")["partition"].nunique().eq(1).all(), \
    "Attempts của cùng task bị chia qua nhiều partition"
assert set(success_metadata["partition"]) <= {
    "train_pool", "validation", "excluded_omnicode"
}


In [2]:
# Kiểm tra chính sách split đang dùng, không lấy test nội bộ từ NVIDIA.
import json

config_path = project_root / "configs/datasets/nvidia_mini_swe_agent_v1_2_action_focused_sft.json"
config = json.loads(config_path.read_text())
phase1_manifest = json.loads((
    project_root / config["local_paths"]["processed"] / "task_selection_manifest.json"
).read_text())
success_report = json.loads((success_dir / "report.json").read_text())
assert "test" not in phase1_manifest["selection"], "Manifest vẫn có test nội bộ"
assert len(phase1_manifest["selection"]["validation"]) == 50
assert config["test_evaluation"]["tasks"] == 500
assert len(success_metadata) == success_report["trajectories"]
assert success_metadata["partition"].value_counts().to_dict() == success_report["partitions"]

validation_keys = {r["task_key"] for r in phase1_manifest["selection"]["validation"]}
assert set(success_metadata.loc[
    success_metadata["partition"].eq("validation"), "task_key"
]) == validation_keys
assert set(success_metadata.loc[
    success_metadata["partition"].eq("train_pool"), "task_key"
]).isdisjoint(validation_keys)



In [3]:
train_metadata = success_metadata[success_metadata["partition"].eq("train_pool")]
train_task_counts = train_metadata.groupby("task_key").size()

train_trajectory_distribution = (
    train_task_counts.value_counts()
    .sort_index()
    .rename_axis("trajectories_per_task")
    .to_frame("tasks")
)
train_trajectory_distribution["trajectories"] = (
    train_trajectory_distribution.index.to_numpy()
    * train_trajectory_distribution["tasks"]
)

assert train_trajectory_distribution["tasks"].sum() == train_metadata["task_key"].nunique()
assert train_trajectory_distribution["trajectories"].sum() == len(train_metadata)
display(train_trajectory_distribution)


,tasks,trajectories
trajectories_per_task,,
1,3163,3163
2,4875,9750
3,12106,36318
4,16,64
5,37,185
6,54,324
